# 🧬 Part 1: DNABERT-2 Embedding Extraction to Disk

This notebook serves as **Part 1** of our modular deep learning workflow. It loads the pre-trained `zhihan1996/DNABERT-2-117M` model and processes the FASTA sequences (SP1, SP2, SP4, and Negative) to extract token-level representations. 

The extracted embeddings and labels are saved as NumPy arrays (`.npy`) on disk, allowing us to train classifiers in separate environments without re-running the heavy foundation model.

### 1. Installation and Repository Setup

In [ ]:
# Install dependencies
!pip install -q transformers einops safetensors huggingface_hub

# Detect environment and clone repository
import os, subprocess, sys

REPO_URL = "https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git"
REPO_NAME = "SP1_TF_Biding_Project"

if os.path.basename(os.getcwd()) == REPO_NAME:
    os.chdir("..")

if not os.path.isdir(REPO_NAME):
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print(f"Repository '{REPO_NAME}' already exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_NAME, "pull"], check=True)

os.chdir(REPO_NAME)
print(f"Current Working Directory: {os.getcwd()}")

# Add path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

### 2. Extract Embeddings
We run our custom extraction script which loads the datasets, instantiates the DNABERT-2 wrapper with all bugfixes (Triton patch, Strategy 2 weights reload), and saves them to disk.

In [ ]:
# Run extraction using the CLI script. Saves data in data/processed/
!python src/extract_embeddings.py --batch_size 64 --max_length 105

### 3. Verify Saved Files
Ensure the files were successfully generated and show their shapes.

In [ ]:
import numpy as np

emb_path = "data/processed/dnabert_embeddings.npy"
lbl_path = "data/processed/dnabert_labels.npy"

if os.path.exists(emb_path) and os.path.exists(lbl_path):
    emb = np.load(emb_path, mmap_mode='r')
    lbl = np.load(lbl_path, mmap_mode='r')
    print("🎉 Verification Succeeded!")
    print(f"  Embeddings shape: {emb.shape} (float16)")
    print(f"  Labels shape:     {lbl.shape}")
else:
    print("❌ Verification Failed. Files not found.")